In [19]:
path = "/Users/andrewweiland/UCCS_REU/Quantum_Circuit_Optimization_Data/trajectories_combined.db"


In [20]:
import sqlite3

In [21]:
conn = sqlite3.connect(path)

In [22]:
import pandas as pd

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """,
    conn
)

print("Table Names:")
print(tables)

Table Names:
                name
0           circuits
1    sqlite_sequence
2         optimizers
3  optimization_runs
4       trajectories
5   trajectory_steps


In [23]:
## All 5 Optimizers used during testing
opt_df = pd.read_sql_query(
    "SELECT * FROM optimizers;",
    conn
)
opt_df

,id,name,runner_type,options_json,description
0,1,wisq_rules,wisq,"{""approx_epsilon"": 0}",WISQ with rules-only optimization (no resynthe...
1,2,wisq_bqskit,wisq,"{""approx_epsilon"": 1e-10, ""opt_timeout"": 300}",WISQ with BQSKit resynthesis
2,3,tket,tket,"{""gate_set"": ""IBMN""}",TKET FullPeepholeOptimise with IBMN gate set
3,4,qiskit_ai,qiskit_ai,"{""optimization_levels"": [3], ""iterations_per_l...",Qiskit AI transpiler at optimization level 3
4,5,qiskit_standard,qiskit_standard,"{""optimization_levels"": [3]}",Standard Qiskit transpiler at optimization lev...


In [24]:
#All individual optimization runs (individual and chains)

traj_df = pd.read_sql_query(
    "SELECT * FROM trajectories;",
    conn
)
traj_df.tail()

,id,circuit_id,chain_name,num_steps,initial_depth,initial_two_qubit_gates,initial_two_qubit_depth,initial_total_gates,final_depth,final_two_qubit_gates,final_two_qubit_depth,final_total_gates,total_duration_seconds,total_reward,improvement_percentage,metadata_json,created_at
16554,16555,1266,chain3_wisq_rules__wisq_bqskit__wisq_bqskit,3,23,8,8,23,71,18,15,127,1202.415318,-122.116532,-125.0,"{""source"": ""chain_3step"", ""step1_run_id"": 1038...",2026-03-05 20:40:44
16555,16556,1266,chain3_wisq_rules__wisq_bqskit__wisq_rules,3,23,8,8,23,83,18,17,133,1501.703982,-152.045398,-125.0,"{""source"": ""chain_3step"", ""step1_run_id"": 1038...",2026-03-05 20:40:44
16556,16557,1267,chain3_wisq_rules__wisq_rules__qiskit_ai,3,23,8,8,23,77,37,58,94,1201.578077,-122.792808,-362.5,"{""source"": ""chain_3step"", ""step1_run_id"": 1038...",2026-03-05 20:40:44
16557,16558,1267,chain3_wisq_rules__wisq_rules__qiskit_standard,3,23,8,8,23,90,51,48,128,1201.367918,-123.331792,-537.5,"{""source"": ""chain_3step"", ""step1_run_id"": 1038...",2026-03-05 20:40:44
16558,16559,1267,chain3_wisq_rules__wisq_rules__tket,3,23,8,8,23,38,21,20,49,1201.609071,-122.155907,-162.5,"{""source"": ""chain_3step"", ""step1_run_id"": 1038...",2026-03-05 20:40:44


In [25]:
#All individual steps taken during optimization runs
traj_steps_df = pd.read_sql_query(
    "SELECT * FROM trajectory_steps;",
    conn
)
traj_steps_df.head()

,id,trajectory_id,step_index,optimizer_id,state_depth,state_two_qubit_gates,state_two_qubit_depth,state_total_gates,state_num_qubits,state_gate_density,...,next_state_time_budget_remaining,reward_improvement_only,reward_efficiency,reward_multi_objective,reward_sparse_final,reward_category_relative,reward_efficiency_normalized,done,duration_seconds,created_at
0,1,1,0,1,17,18,11,78,10,7.8,...,-300.292814,0.000000,-60.039281,-60.145164,0.000000,-60.315262,-0.210098,1,600.292814,2026-03-05 20:39:34
1,2,2,0,1,17,18,11,78,10,7.8,...,-300.192577,0.000000,-60.029258,-60.135140,0.000000,-60.305238,-0.210064,1,600.192577,2026-03-05 20:39:34
2,3,3,0,2,17,18,11,78,10,7.8,...,-102.726940,0.277778,-40.004916,-40.582367,0.277778,-40.390897,0.133535,1,402.726940,2026-03-05 20:39:34
3,4,4,0,2,17,18,11,78,10,7.8,...,-122.451553,0.333333,-41.921822,-42.321822,0.333333,-42.307803,0.182516,1,422.451553,2026-03-05 20:39:34
4,5,5,0,2,17,18,11,78,10,7.8,...,-240.983333,0.333333,-53.775000,-54.051471,0.333333,-54.160981,0.143006,1,540.983333,2026-03-05 20:39:34


In [26]:
# Original Circuits
circuit_df = pd.read_sql_query(
    "SELECT * FROM circuits;",
    conn
)
circuit_df["optimizer_chain"] = circuit_df["name"].str.split("__").str[1:].str.join("__")
circuit_df["original_circuit"] = circuit_df["name"].str.split("__").str[0].str.replace(r"^(artifact_)+", "", regex=True)

circuit_df.tail()

,id,name,category,source,qasm_path,num_qubits,initial_depth,initial_two_qubit_gates,initial_two_qubit_depth,initial_total_gates,gate_density,two_qubit_ratio,created_at,optimizer_chain,original_circuit
2811,2812,artifact_artifact_qft_N006__wisq_rules__tket,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts_s...,6,41,30,24,65,10.833333,0.461538,2026-03-05 20:40:43,wisq_rules__tket,qft_N006
2812,2813,artifact_artifact_qft_N006__wisq_rules__qiskit...,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts_s...,6,102,78,70,133,22.166667,0.586466,2026-03-05 20:40:43,wisq_rules__qiskit_standard,qft_N006
2813,2814,artifact_artifact_qft_N006__wisq_rules__qiskit_ai,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts_s...,6,117,86,77,152,25.333333,0.565789,2026-03-05 20:40:43,wisq_rules__qiskit_ai,qft_N006
2814,2815,artifact_artifact_qft_N006__wisq_rules__wisq_b...,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts_s...,6,109,33,28,230,38.333333,0.143478,2026-03-05 20:40:43,wisq_rules__wisq_bqskit,qft_N006
2815,2816,artifact_artifact_qft_N006__wisq_rules__wisq_r...,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts_s...,6,75,34,29,100,16.666667,0.340000,2026-03-05 20:40:43,wisq_rules__wisq_rules,qft_N006


In [27]:
#All original circuits
local = circuit_df[circuit_df["source"] == "local"]
local

,id,name,category,source,qasm_path,num_qubits,initial_depth,initial_two_qubit_gates,initial_two_qubit_depth,initial_total_gates,gate_density,two_qubit_ratio,created_at,optimizer_chain,original_circuit
0,1,efficient_su2_10_r2,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_10_...,10,17,18,11,78,7.800000,0.230769,2026-03-05 20:39:31,,efficient_su2_10_r2
1,2,efficient_su2_12,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_12....,12,25,66,21,114,9.500000,0.578947,2026-03-05 20:39:31,,efficient_su2_12
2,3,efficient_su2_12_r2,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_12_...,12,19,22,13,94,7.833333,0.234043,2026-03-05 20:39:31,,efficient_su2_12_r2
3,4,efficient_su2_16,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_16....,16,33,120,29,184,11.500000,0.652174,2026-03-05 20:39:31,,efficient_su2_16
4,5,efficient_su2_8,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_8.qasm,8,27,56,21,104,13.000000,0.538462,2026-03-05 20:39:31,,efficient_su2_8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,107,real_amplitudes_8,real_amplitudes,local,benchmarks/ai_transpile/qasm/real_amplitudes_8...,8,15,21,11,53,6.625000,0.396226,2026-03-05 20:39:34,,real_amplitudes_8
107,108,real_amplitudes_8_r2,real_amplitudes,local,benchmarks/ai_transpile/qasm/real_amplitudes_8...,8,12,14,9,38,4.750000,0.368421,2026-03-05 20:39:34,,real_amplitudes_8_r2
108,109,square_heisenberg_N16,square-heisenberg,local,benchmarks/ai_transpile/qasm/square-heisenberg...,16,361,288,144,1024,64.000000,0.281250,2026-03-05 20:39:34,,square_heisenberg_N16
109,110,square_heisenberg_N4,square-heisenberg,local,benchmarks/ai_transpile/qasm/square-heisenberg...,4,121,48,48,172,43.000000,0.279070,2026-03-05 20:39:34,,square_heisenberg_N4


In [28]:
# Circuits Created by Optimization of Other Circuits
artifact = circuit_df[circuit_df["source"] == "artifact"]
artifact

,id,name,category,source,qasm_path,num_qubits,initial_depth,initial_two_qubit_gates,initial_two_qubit_depth,initial_total_gates,gate_density,two_qubit_ratio,created_at,optimizer_chain,original_circuit
111,112,artifact_qft_16__tket,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/q...,16,107,240,58,513,32.062500,0.467836,2026-03-05 20:39:34,tket,qft_16
112,113,artifact_qft_16__qiskit_standard,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/q...,16,255,541,160,1138,71.125000,0.475395,2026-03-05 20:39:34,qiskit_standard,qft_16
113,114,artifact_qft_16__wisq_bqskit,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/q...,16,381,259,80,1790,111.875000,0.144693,2026-03-05 20:39:34,wisq_bqskit,qft_16
114,115,artifact_qft_16__wisq_rules,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/q...,16,341,264,69,1964,122.750000,0.134420,2026-03-05 20:39:34,wisq_rules,qft_16
115,116,artifact_qft_12__tket,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/q...,12,79,132,42,289,24.083333,0.456747,2026-03-05 20:39:34,tket,qft_12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2811,2812,artifact_artifact_qft_N006__wisq_rules__tket,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts_s...,6,41,30,24,65,10.833333,0.461538,2026-03-05 20:40:43,wisq_rules__tket,qft_N006
2812,2813,artifact_artifact_qft_N006__wisq_rules__qiskit...,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts_s...,6,102,78,70,133,22.166667,0.586466,2026-03-05 20:40:43,wisq_rules__qiskit_standard,qft_N006
2813,2814,artifact_artifact_qft_N006__wisq_rules__qiskit_ai,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts_s...,6,117,86,77,152,25.333333,0.565789,2026-03-05 20:40:43,wisq_rules__qiskit_ai,qft_N006
2814,2815,artifact_artifact_qft_N006__wisq_rules__wisq_b...,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts_s...,6,109,33,28,230,38.333333,0.143478,2026-03-05 20:40:43,wisq_rules__wisq_bqskit,qft_N006


In [29]:
sample = artifact.iloc[:100]
sample["optimizer_chain"] = sample["name"].str.split("__").str[1]
sample["original_circuit"] = sample["name"].str.split("__").str[0].str.strip("artifact_")
sample

/var/folders/z7/sn17llg54qn5qwf154c5rpb80000gn/T/ipykernel_38164/457865253.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample["optimizer_chain"] = sample["name"].str.split("__").str[1]
/var/folders/z7/sn17llg54qn5qwf154c5rpb80000gn/T/ipykernel_38164/457865253.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample["original_circuit"] = sample["name"].str.split("__").str[0].str.strip("artifact_")


,id,name,category,source,qasm_path,num_qubits,initial_depth,initial_two_qubit_gates,initial_two_qubit_depth,initial_total_gates,gate_density,two_qubit_ratio,created_at,optimizer_chain,original_circuit
111,112,artifact_qft_16__tket,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/q...,16,107,240,58,513,32.062500,0.467836,2026-03-05 20:39:34,tket,qft_16
112,113,artifact_qft_16__qiskit_standard,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/q...,16,255,541,160,1138,71.125000,0.475395,2026-03-05 20:39:34,qiskit_standard,qft_16
113,114,artifact_qft_16__wisq_bqskit,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/q...,16,381,259,80,1790,111.875000,0.144693,2026-03-05 20:39:34,wisq_bqskit,qft_16
114,115,artifact_qft_16__wisq_rules,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/q...,16,341,264,69,1964,122.750000,0.134420,2026-03-05 20:39:34,wisq_rules,qft_16
115,116,artifact_qft_12__tket,qft,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/q...,12,79,132,42,289,24.083333,0.456747,2026-03-05 20:39:34,tket,qft_12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,207,artifact_barenco_tof_10__tket,feynman,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/b...,19,428,178,162,588,30.947368,0.302721,2026-03-05 20:39:36,tket,barenco_tof_10
207,208,artifact_barenco_tof_10__qiskit_standard,feynman,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/b...,19,635,325,298,879,46.263158,0.369738,2026-03-05 20:39:36,qiskit_standard,barenco_tof_10
208,209,artifact_barenco_tof_10__qiskit_ai,feynman,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/b...,19,12,0,0,98,5.157895,0.000000,2026-03-05 20:39:36,qiskit_ai,barenco_tof_10
209,210,artifact_barenco_tof_10__wisq_bqskit,feynman,artifact,/home/lbaird/repos/qdev-laura/data/artifacts/b...,19,458,156,143,637,33.526316,0.244898,2026-03-05 20:39:36,wisq_bqskit,barenco_tof_10


In [69]:
## Creates df with all calculated chains
## Do not trust data for initial. 
## It likely is using info from intermediate step, not initial circuit.

import numpy as np
import json
master_df = pd.DataFrame(columns=[
    "id",
    "parent_circuit_id",
    "parent_circuit",
    "original_circuit_id",
    "original_circuit",
    "original_category",
    "original_circuit_path",
    "chain_name",
    "opt_level",
    "opt_1",
    "opt_2",
    "opt_3",
    "qasm_path",
    "initial_num_qubits",
    "initial_depth",
    "initial_two_qubit_gates",
    "initial_total_gates",
    "final_num_qubits",
    "final_depth",
    "final_two_qubit_gates",
    "final_total_gates",
    "total_runtime_seconds"
])

single_dic = {"single_1": "wisq_rules", "single_2": "wisq_bqskit", "single_3": "tket", "single_4": "qiskit_ai", "single_5": "qiskit_standard"}

for index, row in traj_df.iterrows():
    parent_circuit_id = -1
    original_circuit_id = -1
    parent_circuit = ""
    chain_name = ""
    original_circuit = ""
    opt_level = -1
    opt_1 = "N/A"
    opt_2 = "N/A"
    opt_3 = "N/A"
    qasm_path = "N/A"
    initial_num_qubits = -1
    initial_depth = -1
    initial_two_qubit_gates = 0
    initial_total_gates = 0
    final_num_qubits = 0
    final_depth = 0
    final_two_qubit_gates = 0
    final_total_gates = 0

    
    metadata = json.loads(row["metadata_json"])

    parent_circuit_id = row["circuit_id"]
    chain_name = row["chain_name"]
    
    #final_num_qubits       = row["final_num_qubits"]
    final_depth             = row["final_depth"]
    final_two_qubit_gates   = row["final_two_qubit_gates"]
    final_total_gates       = row["final_total_gates"]
    original_circuit        = circuit_df.loc[circuit_df["id"] == row["circuit_id"]]["original_circuit"].values[0]
    original_circuit_id     = circuit_df.loc[circuit_df["name"] == original_circuit]["id"].values[0]
    original_category       = local.loc[local["name"] == original_circuit]["category"].values[0]
    original_circuit_path   = local.loc[local["name"] == original_circuit]["qasm_path"].values[0]


    initial_num_qubits      = local.loc[local["name"] == original_circuit]["num_qubits"].values[0]
    initial_depth           = local.loc[local["name"] == original_circuit]["initial_depth"].values[0]
    initial_two_qubit_gates = local.loc[local["name"] == original_circuit]["initial_two_qubit_gates"].values[0]
    initial_total_gates     = local.loc[local["name"] == original_circuit]["initial_total_gates"].values[0]
    total_runtime              = row["total_duration_seconds"]
    
    if "single" in chain_name:
        opt_1 = single_dic.get(chain_name, "N/A") 
        chain_name = single_dic.get(chain_name, "N/A")
        opt_level = 1
    else:        
        opt_1 = metadata.get("step1_optimizer", "N/A")
        opt_2 = metadata.get("step2_optimizer", "N/A")
        opt_3 = metadata.get("step3_optimizer", "N/A")
        parent_circuit = metadata.get("original_circuit")
        if opt_3 != "N/A":
            opt_level = 3
        elif opt_2 != "N/A":
            opt_level = 2
        else:
            opt_level = 1
        

    if parent_circuit == "":
        parent_circuit = original_circuit
            

    

    master_df.loc[len(master_df)] = [
        index + 1,
        parent_circuit_id,
        parent_circuit,
        original_circuit_id,
        original_circuit,
        original_category,
        original_circuit_path,
        chain_name,
        opt_level,
        opt_1,
        opt_2,
        opt_3,
        qasm_path,
        initial_num_qubits,
        initial_depth,
        initial_two_qubit_gates,
        initial_total_gates,
        final_num_qubits,
        final_depth,
        final_two_qubit_gates,
        final_total_gates,
        total_runtime
    ]


master_df = master_df.replace("N/A", np.nan)
## Adds entire circuit path in format original_circuit__opt1__opt2__opt3
master_df["opt_chain"] = master_df.apply(
    lambda row: "__".join(
        [str(x) for x in [
            row["original_circuit"],
            row["opt_1"],
            row["opt_2"],
            row["opt_3"]
        ] if pd.notna(x)]
    ),
    axis=1
)


        

/var/folders/z7/sn17llg54qn5qwf154c5rpb80000gn/T/ipykernel_38164/3730343288.py:125: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  master_df = master_df.replace("N/A", np.nan)


In [112]:
# Shows how many unique chains from each starting circuit
# Should have 155 if all have been completed

unique_counts = (
    master_df
    .drop_duplicates(subset=["original_circuit", "opt_1", "opt_2", "opt_3"])
    .groupby("original_circuit")
    .size()
    .sort_index()
)

print(unique_counts)

original_circuit
W-state            81
adder              93
barenco_tof_10     30
barenco_tof_3     155
barenco_tof_4     154
                 ... 
tof_10             30
tof_3              55
tof_4              55
tof_5              55
vbe_adder_3        55
Length: 106, dtype: int64


In [115]:
#All original circuits
og_circuits = local["name"].dropna().unique().tolist()
#All optimizers used
optimizers = ["wisq_rules", "wisq_bqskit", "tket", "qiskit_ai", "qiskit_standard"]

##All computed chains
unique_opt_chains = master_df["opt_chain"].dropna().unique().tolist()

In [116]:
og_circuits_master = master_df["original_circuit"].dropna().unique().tolist()
len(og_circuits_master)

106

In [117]:
#Finds all possible chains for each circuit

from itertools import product
all_possible_chains = []

for circuit in og_circuits:
    for chain_length in range(1, 4):
        for chain in product(optimizers, repeat=chain_length):
            opt_chain = "__".join([circuit] + list(chain))
            all_possible_chains.append(opt_chain)

print(len(all_possible_chains))
print(all_possible_chains[-10:])

17205
['square_heisenberg_N9__qiskit_standard__qiskit_ai__wisq_rules', 'square_heisenberg_N9__qiskit_standard__qiskit_ai__wisq_bqskit', 'square_heisenberg_N9__qiskit_standard__qiskit_ai__tket', 'square_heisenberg_N9__qiskit_standard__qiskit_ai__qiskit_ai', 'square_heisenberg_N9__qiskit_standard__qiskit_ai__qiskit_standard', 'square_heisenberg_N9__qiskit_standard__qiskit_standard__wisq_rules', 'square_heisenberg_N9__qiskit_standard__qiskit_standard__wisq_bqskit', 'square_heisenberg_N9__qiskit_standard__qiskit_standard__tket', 'square_heisenberg_N9__qiskit_standard__qiskit_standard__qiskit_ai', 'square_heisenberg_N9__qiskit_standard__qiskit_standard__qiskit_standard']


In [118]:
# All original Circuits that had no Optimization done
missing_circuits  = list(set(og_circuits) - set(og_circuits_master))

missing_circuits


['teleportv2', 'inverseqft1', 'qec', 'teleport', 'inverseqft2']

In [119]:
##All missing chains
missing_chains = list(set(all_possible_chains) - set(unique_opt_chains))

In [120]:
# Number of missing chains for each chain length
missing_one = 0
missing_two = 0
missing_three = 0
for string in missing_chains:
    split = string.split("__")
    if len(split) == 4:
        missing_three += 1
    elif len(split) == 3:
        missing_two += 1
    elif len(split) == 2:
        missing_one += 1

print(f"Missing one optimizer chains: {missing_one}")
print(f"Missing two optimizer chains: {missing_two}")
print(f"Missing three optimizer chains: {missing_three}")


Missing one optimizer chains: 43
Missing two optimizer chains: 287
Missing three optimizer chains: 9970


In [128]:
master_df.to_csv("master.csv", index=False)

In [38]:
print(traj_df.iloc[1000]["metadata_json"])
traj_df.iloc[1000]

{"source": "synthesized_from_optimization_runs", "run_id": 868}


id                                                                      1001
circuit_id                                                                73
chain_name                                                          single_4
num_steps                                                                  1
initial_depth                                                             68
initial_two_qubit_gates                                                   84
initial_two_qubit_depth                                                   33
initial_total_gates                                                      210
final_depth                                                              113
final_two_qubit_gates                                                    119
final_two_qubit_depth                                                     99
final_total_gates                                                        245
total_duration_seconds                                              2.166294

In [37]:
local[local["name"] == "efficient_su2_16"]

,id,name,category,source,qasm_path,num_qubits,initial_depth,initial_two_qubit_gates,initial_two_qubit_depth,initial_total_gates,gate_density,two_qubit_ratio,created_at,optimizer_chain,original_circuit
3,4,efficient_su2_16,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_16....,16,33,120,29,184,11.5,0.652174,2026-03-05 20:39:31,,efficient_su2_16


In [51]:
local[local["id"] == 73]

,id,name,category,source,qasm_path,num_qubits,initial_depth,initial_two_qubit_gates,initial_two_qubit_depth,initial_total_gates,gate_density,two_qubit_ratio,created_at,optimizer_chain,original_circuit
72,73,qft_N009,qft,local,benchmarks/ai_transpile/qasm/qft/qft_N009.qasm,9,68,84,33,210,23.333333,0.4,2026-03-05 20:39:33,,qft_N009


In [70]:
master_df[master_df["original_circuit_id"] != master_df["parent_circuit_id"]]

,id,parent_circuit_id,parent_circuit,original_circuit_id,original_circuit,original_category,original_circuit_path,chain_name,opt_level,opt_1,...,initial_num_qubits,initial_depth,initial_two_qubit_gates,initial_total_gates,final_num_qubits,final_depth,final_two_qubit_gates,final_total_gates,total_runtime_seconds,opt_chain
1450,1451,264,W-state,6,W-state,feynman,benchmarks/ai_transpile/qasm/feynman/W-state.qasm,chain_qiskit_ai__qiskit_ai,2,qiskit_ai,...,3,6,2,9,0,6,3,9,0.379314,W-state__qiskit_ai__qiskit_ai
1451,1452,264,W-state,6,W-state,feynman,benchmarks/ai_transpile/qasm/feynman/W-state.qasm,chain_qiskit_ai__qiskit_standard,2,qiskit_ai,...,3,6,2,9,0,27,6,42,0.187112,W-state__qiskit_ai__qiskit_standard
1452,1453,264,W-state,6,W-state,feynman,benchmarks/ai_transpile/qasm/feynman/W-state.qasm,chain_qiskit_ai__tket,2,qiskit_ai,...,3,6,2,9,0,22,6,30,0.870331,W-state__qiskit_ai__tket
1453,1454,263,W-state,6,W-state,feynman,benchmarks/ai_transpile/qasm/feynman/W-state.qasm,chain_qiskit_standard__qiskit_ai,2,qiskit_standard,...,3,6,2,9,0,27,6,42,0.210880,W-state__qiskit_standard__qiskit_ai
1454,1455,263,W-state,6,W-state,feynman,benchmarks/ai_transpile/qasm/feynman/W-state.qasm,chain_qiskit_standard__qiskit_standard,2,qiskit_standard,...,3,6,2,9,0,26,6,40,0.034661,W-state__qiskit_standard__qiskit_standard
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16554,16555,1266,mod5_4,28,mod5_4,feynman,benchmarks/ai_transpile/qasm/feynman/mod5_4.qasm,chain3_wisq_rules__wisq_bqskit__wisq_bqskit,3,wisq_rules,...,5,23,4,23,0,71,18,127,1202.415318,mod5_4__wisq_rules__wisq_bqskit__wisq_bqskit
16555,16556,1266,mod5_4,28,mod5_4,feynman,benchmarks/ai_transpile/qasm/feynman/mod5_4.qasm,chain3_wisq_rules__wisq_bqskit__wisq_rules,3,wisq_rules,...,5,23,4,23,0,83,18,133,1501.703982,mod5_4__wisq_rules__wisq_bqskit__wisq_rules
16556,16557,1267,mod5_4,28,mod5_4,feynman,benchmarks/ai_transpile/qasm/feynman/mod5_4.qasm,chain3_wisq_rules__wisq_rules__qiskit_ai,3,wisq_rules,...,5,23,4,23,0,77,37,94,1201.578077,mod5_4__wisq_rules__wisq_rules__qiskit_ai
16557,16558,1267,mod5_4,28,mod5_4,feynman,benchmarks/ai_transpile/qasm/feynman/mod5_4.qasm,chain3_wisq_rules__wisq_rules__qiskit_standard,3,wisq_rules,...,5,23,4,23,0,90,51,128,1201.367918,mod5_4__wisq_rules__wisq_rules__qiskit_standard


In [72]:
master_df.to_csv("master.csv", index=False)